# Student-CNN: Pure Convolutional Student Network for LoFTR Distillation

This notebook implements **Student-CNN**, a fully convolutional student network that replaces LoFTR's attention mechanism with dilated convolutions and cross-correlation.

## Architecture Overview

| Component | Teacher (LoFTR) | Student-CNN |
|---|---|---|
| Backbone | ResNet-18 + FPN (~11M params) | Lightweight ResNet (~1M params) |
| Coarse features | 256-dim, 1/8 res | 128-dim, 1/8 res |
| Fine features | 128-dim, 1/2 res | 64-dim, 1/2 res |
| Feature interaction | 4 self+cross attention layers | Dilated convs (rates 1,2,4,8) |
| Coarse matching | Dual-softmax | Dual-softmax (same) |
| Fine refinement | Local window correlation | Local window correlation (same) |

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
import numpy as np

from models import StudentCNN, StudentCNNConfig
from losses import DistillationLoss

## 1. Model Configuration

All hyperparameters are centralized in `StudentCNNConfig`. Key settings:
- Backbone channels: [32, 64, 128]
- Dilated conv rates: [1, 2, 4, 8] → ~248px effective receptive field at original resolution
- Dual-softmax temperature: 0.1
- Fine refinement window: 5×5

In [ ]:
config = StudentCNNConfig()

print("=== Student-CNN Configuration ===")
print(f"Backbone channels:     {config.backbone_channels}")
print(f"Coarse feature dim:    {config.coarse_dim} (teacher: {config.teacher_coarse_dim})")
print(f"Fine feature dim:      {config.fine_dim} (teacher: {config.teacher_fine_dim})")
print(f"Dilation rates:        {config.dilation_rates}")
print(f"Temperature:           {config.temperature}")
print(f"Match threshold:       {config.match_threshold}")
print(f"Fine window size:      {config.fine_window_size}")
print(f"\nDistillation weights:")
print(f"  λ_coarse_kd = {config.lambda_coarse_kd}")
print(f"  λ_feat_kd   = {config.lambda_feat_kd}")
print(f"  λ_fine_kd   = {config.lambda_fine_kd}")
print(f"  λ_gt        = {config.lambda_gt}")

## 2. Instantiate Model

The Student-CNN model consists of:
1. **LightweightBackbone**: 3-stage ResNet with ~1M params
2. **DilatedInteractionModule**: 4 dilated conv layers (shared weights for both images)
3. **CoarseMatching**: Dual-softmax confidence matrix + mutual NN extraction
4. **FineRefinement**: Local window correlation with spatial expectation
5. **feat_projector**: 1×1 conv for distillation (128 → 256 channels, training only)

In [ ]:
model = StudentCNN(config)

# Parameter count per component
param_counts = model.count_parameters()
print("=== Parameter Count ===")
for name, count in param_counts.items():
    print(f"  {name:20s}: {count:>10,d} ({count/1e6:.2f}M)")

print(f"\nNote: feat_projector is only used during training (discarded at inference)")
inference_params = param_counts['total'] - param_counts['feat_projector']
print(f"Inference parameters: {inference_params:,d} ({inference_params/1e6:.2f}M)")

## 3. Forward Pass Test (Training Mode)

In training mode, the model outputs:
- `conf_matrix`: (B, N0, N1) coarse confidence matrix
- `sim_matrix`: (B, N0, N1) raw similarity scores
- `student_coarse_feat0/1`: (B, 128, H/8, W/8) raw coarse features
- `student_coarse_feat0/1_proj`: (B, 256, H/8, W/8) projected features for distillation

In [ ]:
# Create dummy input (batch of 2 image pairs, 480×640)
B, H, W = 2, 480, 640
data_train = {
    'image0': torch.randn(B, 1, H, W),
    'image1': torch.randn(B, 1, H, W),
}

# Training mode forward pass
model.train()
with torch.no_grad():
    out_train = model(data_train)

print("=== Training Mode Outputs ===")
for key in ['conf_matrix', 'sim_matrix',
            'student_coarse_feat0', 'student_coarse_feat1',
            'student_coarse_feat0_proj', 'student_coarse_feat1_proj']:
    if key in out_train:
        v = out_train[key]
        print(f"  {key:35s}: {str(v.shape):20s} (dtype={v.dtype})")

print(f"\nCoarse feature map size: {out_train['hw0_c']}")
print(f"Confidence matrix shape: {out_train['conf_matrix'].shape}")
print(f"Confidence range: [{out_train['conf_matrix'].min():.6f}, {out_train['conf_matrix'].max():.6f}]")

## 4. Forward Pass Test (Inference Mode)

In inference mode, the model additionally outputs:
- `keypoints0/1`: (M, 2) matched keypoint coordinates at original resolution
- `confidence`: (M,) match confidence scores

This is compatible with the LoFTR teacher's output interface.

In [ ]:
# Inference mode
data_eval = {
    'image0': torch.randn(1, 1, H, W),
    'image1': torch.randn(1, 1, H, W),
}

model.eval()
with torch.no_grad():
    out_eval = model(data_eval)

print("=== Inference Mode Outputs ===")
print(f"  keypoints0 shape: {out_eval['keypoints0'].shape}")
print(f"  keypoints1 shape: {out_eval['keypoints1'].shape}")
print(f"  confidence shape: {out_eval['confidence'].shape}")
n_matches = len(out_eval['keypoints0'])
print(f"\nNumber of matches found: {n_matches}")

if n_matches > 0:
    print(f"\nFirst 5 matches (keypoints0 → keypoints1):")
    for i in range(min(5, n_matches)):
        kp0 = out_eval['keypoints0'][i].numpy()
        kp1 = out_eval['keypoints1'][i].numpy()
        conf = out_eval['confidence'][i].item()
        print(f"  ({kp0[0]:.1f}, {kp0[1]:.1f}) → ({kp1[0]:.1f}, {kp1[1]:.1f})  conf={conf:.4f}")

## 5. Distillation Loss

Multi-level distillation loss with 4 components:

| Loss | Target | Method | Weight |
|---|---|---|---|
| L_coarse_kd | Teacher's dual-softmax confidence | KL divergence | λ₁=1.0 |
| L_feat_kd | Intermediate feature representations | MSE + 1×1 conv projector | λ₂=0.5 |
| L_fine_kd | Sub-pixel refined coordinates | L1 distance | λ₃=0.5 |
| L_gt | GT correspondences (depth + camera pose) | Focal loss + L2 | λ₄=1.0 |

In [ ]:
loss_fn = DistillationLoss(config)

# Simulate teacher outputs for testing loss computation
model.train()
data_train = {
    'image0': torch.randn(B, 1, H, W),
    'image1': torch.randn(B, 1, H, W),
}

with torch.no_grad():
    student_out = model(data_train)

# Mock teacher outputs (in practice, these come from frozen LoFTR)
H_c, W_c = student_out['hw0_c']
teacher_out = {
    'conf_matrix': torch.rand(B, H_c * W_c, H_c * W_c),
    'teacher_coarse_feat0': torch.randn(B, config.teacher_coarse_dim, H_c, W_c),
    'teacher_coarse_feat1': torch.randn(B, config.teacher_coarse_dim, H_c, W_c),
}

losses = loss_fn(student_out, teacher_out)

print("=== Distillation Loss Components ===")
for name, val in losses.items():
    print(f"  {name:15s}: {val.item():.6f}")

## 6. Model Architecture Summary

Full model architecture for reference.

In [ ]:
print(model)

## 7. Comparison with LoFTR Teacher

The key experimental question: **Can dilated convolutions (large receptive field) + simple dot-product correlation replace attention-based feature interaction?**

### What Student-CNN preserves from LoFTR:
- Coarse-to-fine matching pipeline
- Dual-softmax confidence matrix
- Local window fine refinement

### What Student-CNN replaces:
- ❌ Self/cross-attention layers → ✅ Dilated convolutions (rates 1,2,4,8)
- ❌ ResNet-18 FPN backbone → ✅ Lightweight 3-stage ResNet
- ❌ 256-dim coarse features → ✅ 128-dim (with 1×1 projector for distillation)

### Expected trade-offs:
- ✅ Significantly fewer parameters and FLOPs
- ✅ Faster inference (no quadratic attention)
- ❓ Performance drop on low-texture / large viewpoint change scenes (where attention helps most)

In [ ]:
# FLOPs estimation (approximate)
try:
    from thop import profile
    model.eval()
    dummy_data = {'image0': torch.randn(1, 1, 480, 640), 'image1': torch.randn(1, 1, 480, 640)}
    flops, params = profile(model, inputs=(dummy_data,), verbose=False)
    print(f"FLOPs:  {flops/1e9:.2f} GFLOPs")
    print(f"Params: {params/1e6:.2f} M")
except ImportError:
    print("Install 'thop' package for FLOPs estimation: pip install thop")
    param_counts = model.count_parameters()
    print(f"Total params: {param_counts['total']/1e6:.2f} M")
    print(f"Inference params: {(param_counts['total'] - param_counts['feat_projector'])/1e6:.2f} M")